# 03 — Evaluation: all three conditions

Runs base / SFT-only / distilled students through the agentic harness at chain lengths 1, 3, 5 (the synthetic task set — `harness/tasks.py::load_tasks(chain_length)`, default `source="synthetic"`; see README's "Why synthetic tasks for multi-step eval"), plus the general-LM perplexity baseline. `load_tasks(1, source="glaive_test")` is available as a secondary, single-step-only, in-distribution data point but isn't part of the chain-length comparison itself.

Requires, before running this notebook:
- `notebooks/02_training.ipynb` to have produced all three conditions' checkpoints
- `python -m adbench.data.general_eval` to have produced `data/general_eval/wikitext2_sample.jsonl`

This notebook only orchestrates — all the actual logic (config resolution, the harness-run loop, the model_fn wiring, perplexity, metrics, output writing) lives in `adbench.evaluation.run_eval`/`.metrics`/`.perplexity` and is unit-tested locally (`tests/test_run_eval.py`, `tests/test_metrics.py`, `tests/test_perplexity.py`) without a GPU — only the actual checkpoint loading needs one.

In [ ]:
# Private repo: create a GitHub personal access token (repo scope) and
# add it as a Colab secret named GH_TOKEN (key icon in the left sidebar,
# then enable notebook access for it) before running this cell.
# Skip this cell if this notebook's runtime already has the repo cloned
# (e.g. you ran 00_setup_colab.ipynb in this same session).
import os

if not os.path.isdir('/content/agentic-distillation-benchmark'):
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
    os.environ['GH_TOKEN'] = token
    !git clone https://$GH_TOKEN@github.com/Nahla-Nabil/agentic-distillation-benchmark.git

%cd /content/agentic-distillation-benchmark
!pip install -q -r requirements-colab.txt
!pip install -q -e .

# Runs ALL THREE conditions x all three chain lengths + perplexity in one pass
# (equivalent to `python -m adbench.evaluation.run_eval` from the CLI):
from adbench.evaluation.run_eval import evaluate_condition, write_eval_outputs
from adbench.training.train import CONDITIONS, REPO_ROOT, load_experiment_config

experiment_config = load_experiment_config("configs/experiment.yaml")
models_config = load_experiment_config("configs/models.yaml")
chain_lengths = experiment_config["harness"]["chain_lengths"]

all_rows = []
perplexities = {}
for condition in CONDITIONS:
    print(f"=== Evaluating {condition} ===")
    rows, perplexity = evaluate_condition(condition, experiment_config, models_config, chain_lengths)
    all_rows.extend(rows)
    perplexities[condition] = perplexity
    print(f"{condition}: {len(rows)} tasks run, perplexity={perplexity:.2f}")

output = write_eval_outputs(all_rows, perplexities, REPO_ROOT / "results")
print("wrote results/eval_results.jsonl, eval_results.csv, eval_summary.json")

## Results

`output["summary"]` (one entry per condition x chain length) already has everything below — this just tabulates/plots it. See `evaluation/metrics.py`'s module docstring for exactly what each field means, especially the "DESIGN DECISION" section on how a step that succeeds after a retry is credited.

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(output["summary"])
summary_df[["condition", "chain_length", "n_tasks", "full_chain_success_rate",
            "per_step_success_rate", "clean_step_success_rate", "recovery_rate", "perplexity"]]

### Success rate vs chain length — the key figure for the research question

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for condition in CONDITIONS:
    sub = summary_df[summary_df["condition"] == condition].sort_values("chain_length")
    ax.plot(sub["chain_length"], sub["full_chain_success_rate"], marker="o", label=condition)
ax.set_xlabel("chain length")
ax.set_ylabel("full-chain success rate")
ax.set_xticks(chain_lengths)
ax.set_ylim(0, 1.05)
ax.set_title("Does multi-step success degrade faster than single-step?")
ax.legend()
plt.tight_layout()
plt.show()

### Error breakdown per condition (protocol vs tool-execution — `harness/errors.py`)

In [ ]:
from adbench.evaluation.metrics import error_category_breakdown

for condition in CONDITIONS:
    condition_rows = [r for r in all_rows if r["condition"] == condition]
    print(condition, error_category_breakdown(condition_rows))